middleware

in airport we we reach aurport than do check in(this check in a middleware).

example we can have a summarisation middleware where we can say that after 10 messages we summarrise the messages and this is done by the llm 

some more examples of middleware...human in the feedback,model call limit 
![alt text](image.png)


###summarisation on message length 


In [1]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

# 1. Configure SummarizationMiddleware (triggers at 10 messages, keeps last 4)
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
    checkpointer=InMemorySaver(),
)

In [2]:
config = {"configurable": {"thread_id": "test-1"}}

questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"{q}  -->  messages: {len(response['messages'])}")

What is 2+2?  -->  messages: 2
What is 10*5?  -->  messages: 4
What is 100/4?  -->  messages: 6
What is 15-7?  -->  messages: 8
What is 3*3?  -->  messages: 10
What is 4*4?  -->  messages: 6


In [3]:
print("FIRST MESSAGE (THE SUMMARY):\n")
print(response["messages"][0].content)

FIRST MESSAGE (THE SUMMARY):

Here is a summary of the conversation to date:

## SESSION INTENT

The user is asking a series of basic arithmetic questions.

## SUMMARY

- The user asked "What is 2+2?", and the AI correctly answered "4".
- The user asked "What is 10*5?", and the AI correctly answered "50".
- The user asked "What is 100/4?", and the AI correctly answered "25".
- The user asked "What is 15-7?".

## ARTIFACTS

None

## NEXT STEPS

- Answer the user's current question: "What is 15-7?" (Answer: 8).


based on tokens 


In [4]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(override=True)

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

# Trigger summarization at 550 tokens, keeping the most recent 200 tokens
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search_hotels],
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("tokens", 550),
            keep=("tokens", 200),
        )
    ],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "test-1"}}

# Helper function to count approximate tokens in the current conversation
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # ~4 characters per token

In [5]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")

print("\nGenerated Summary:\n", response["messages"][0].content)

Paris: ~170 tokens, 4 messages
London: ~342 tokens, 8 messages
Tokyo: ~513 tokens, 12 messages
New York: ~536 tokens, 9 messages
Dubai: ~592 tokens, 9 messages
Singapore: ~591 tokens, 6 messages

Generated Summary:
 Here is a summary of the conversation to date:

## SESSION INTENT

The user wants to find hotels in multiple cities: Paris, London, Tokyo, New York, and Dubai.

## SUMMARY

- Previously searched for and presented hotel options in Paris, London, Tokyo, and New York.
- Received a new request to find hotels in Dubai.
- Executed a hotel search for "Dubai" using the `search_hotels` tool.
- Retrieved and presented Dubai hotel options: Grand Hotel ($350/night), City Inn ($180/night), and Budget Stay ($75/night).

## ARTIFACTS

- Performed hotel search tool calls for "Paris", "London", "Tokyo", "New York", and "Dubai".

## NEXT STEPS

- Wait for the user to select hotels from the provided lists or request hotel searches for additional cities.


human in the loop middleware 

In [ ]:
this middle ware will ask you question before doing anything so it needs human intervention in middle

In [6]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv(override=True)

@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {recipient} with subject '{subject}'."

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[send_email],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": {"allowed_decisions": ["approve", "edit", "reject"]}
            }
        )
    ],
)

In [7]:
config = {"configurable": {"thread_id": "thread-approve"}}

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Send an email to john@test.com saying Hello, how are you?"}
        ]
    },
    config=config,
)

# Inspect the paused interrupt details
print(result["__interrupt__"])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'Hello, how are you?'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'Hello, how are you?'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='252353fb1ce1dc1a815621db90655b34', response_schema=None)]


In [8]:
approved_result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
)

print(approved_result["messages"][-1].text)

Email sent to john@test.com with subject 'Hello'.


In [9]:
config_edit = {"configurable": {"thread_id": "thread-edit"}}

# Step 1: Trigger interrupt
agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Send an email to john@test.com saying Hello, how are you?"}
        ]
    },
    config=config_edit,
)

# Step 2: Edit the arguments and resume
edited_result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email",
                        "args": {
                            "recipient": "john.updated@test.com",
                            "subject": "Updated Meeting",
                            "body": "Hello John, let's meet tomorrow at 10 AM.",
                        },
                    },
                }
            ]
        }
    ),
    config=config_edit,
)

print(edited_result["messages"][-1].text)

The email has been successfully sent to john.updated@test.com with the subject "Updated Meeting" (as modified by the system review).


In [10]:
config_reject = {"configurable": {"thread_id": "thread-reject"}}

# Step 1: Trigger interrupt
agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Send an email to john@test.com saying Hello, how are you?"}
        ]
    },
    config=config_reject,
)

# Step 2: Reject the tool call with a reason
rejected_result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": "Do not send this email right now.",
                }
            ]
        }
    ),
    config=config_reject,
)

print(rejected_result["messages"][-1].text)

The email could not be sent because you rejected the request.
